# 4. ONNX export

Base/standard Tiny export the image encoder, dynamic-prefix decoder, and optional refinement graph. Fast Tiny exports one complete graph with encoder, parallel prediction and two refinements. Both adapters verify FP32 outputs before exporting.

## Model selection

Read [0_model_options.ipynb](0_model_options.ipynb) for Base, Tiny and the custom Fast Tiny recipe. Choose `configs/alpr_base.yaml`, `configs/alpr_tiny.yaml`, or `configs/alpr_fast_tiny.yaml` via `CONFIG_PATH`. Keep that choice consistent across notebooks and use separate output folders.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "parseq_ocr").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "parseq_ocr").is_dir():
    raise RuntimeError("Open this notebook from the repository or notebooks directory")
sys.path.insert(0, str(ROOT))
from parseq_ocr.config import load_config
# Dummy presets: configs/dummy_base.yaml, dummy_tiny.yaml, dummy_fast_tiny.yaml.
# Real-data presets: configs/alpr_base.yaml, alpr_tiny.yaml, alpr_fast_tiny.yaml.
# PARSEQ_CONFIG overrides this default when it is set.
CONFIG_PATH = os.environ.get("PARSEQ_CONFIG", "configs/dummy_tiny.yaml")
config = load_config(CONFIG_PATH)
print("Config:", CONFIG_PATH)
print("Model:", config["model"], "input:", config["image_size"])
print("Artifacts:", config["output_dir"])


In [ ]:
from parseq_ocr.export import export_onnx

deployment = export_onnx(config)
print("Exported:", [p.name for p in deployment.glob("*.onnx")])

In [ ]:
import json
import onnx

for path in deployment.glob("*.onnx"):
    onnx.checker.check_model(str(path))
print(json.loads((deployment / "export_parity.json").read_text()))
print(json.loads((deployment / "metadata.json").read_text())["decoding"])

Batch size is dynamic. Standard split graphs accept dynamic decoder prefix lengths; Fast Tiny embeds its fixed parallel-query capacity in one graph. Input geometry is fixed for each export (64×256 for standard presets, 48×192 for Fast Tiny). A different geometry requires a matching checkpoint and a new export. The synthetic parity check verifies the adapter math; notebook 7 checks accuracy on real held-out images.